# 07 – Lexicon v1.1 sign-off package

Everything the team needs to decide whether lexicon v1.1 is frozen: version tag and changelog, confirmation that the audit tables use the post-fix numbers, the terms that never fire, candidate missing terms found by reading low-hit news chunks, the AI-but-not-regulation articles, and one review file with all proposals.

**Rules for this notebook**
- **The lexicon's terms are not changed here.** Only metadata is added (date, changelog, sign-off flag), and the notebook asserts that the frames, ambiguous list and jurisdiction list are identical before and after. All proposed changes go to review files.
- **No hypothesis test (H1–H3) is run.** The last cell checks that no H1 result file exists and that the gates in notebooks 05 and 06 are still closed.
- The reading of news chunks in section 4 was done by the AI assistant that built this notebook, on 2026-10-27, from the chunks listed in `data/lexicon_review_lowhit_sample.csv`. It is a proposal list for human review, not an exhaustive or validated one.

In [1]:
import hashlib, json, re
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

from framescore import Lexicon, find_hits, count_vector, cut, lem, build_matcher, nlp

CFG = json.load(open("config.json"))
OUT = Path(CFG["analysis_dir"]); FR = Path("dataset/frames"); REF = FR / "reference"
SEED = 42
LEX_PATH = Path(CFG["lexicon_json"])
LEX = Lexicon(LEX_PATH); FRAMES = LEX.frames

## 1. Version tag, date and changelog
v1.0 (the version the team first reviewed) is kept in `dataset/frames/reference/frame_keywords_v1.0.json`. The changelog is built by comparing v1.0 with v1.1 on lemmas, so it cannot miss a term; every difference must have a reason below or the cell stops.

In [2]:
v0 = json.load(open(REF / "frame_keywords_v1.0.json")); v1 = json.load(open(LEX_PATH))
lemset = lambda d: {fr: {lem(t) for t in ts} for fr, ts in d["frames"].items()}
L0, L1 = lemset(v0), lemset(v1)

REASONS = {
 # removed
 "red tape": "Pejorative: measures stance, not topic, which would make H2 partly circular (rare: 2 hits).",
 "bureaucracy": "Pejorative: measures stance, not topic (rare: 8 hits).",
 "bias": "Mostly political or model bias, a second meaning; the term with the most hits in the frame.",
 "alignment": "Often 'policy alignment', not AI alignment.",
 "verify": "Mostly age or identity verification, not misinformation.",
 "verification": "Mostly age or identity verification, not misinformation.",
 "manipulate": "Generic; second sense.", "autonomy": "Generic; second sense (human agency).",
 "minority": "Generic; second sense.", "licence": "Second sense (data licensing).", "license": "Second sense (data licensing).",
 "agency": "Often 'human agency'; also appears in the Q7 prompt (prompt echo).",
 "gdpr": "Moved to jurisdiction_terms so the region comparison (H3) is not circular.",
 "scientific progress": "Sub-phrase covered by 'progress' and by longest-match filtering.",
 "technological progress": "Sub-phrase covered by 'progress' and by longest-match filtering.",
 "technological advancement": "Sub-phrase covered by 'advancement' and by longest-match filtering.",
 "legal framework": "Sub-phrase covered by 'framework' and by longest-match filtering.",
 # added
 "copyright": "Topic with no home in v1.0 (rights).", "intellectual property": "Topic with no home in v1.0 (rights).",
 "creator": "Topic with no home in v1.0 (rights).", "child": "Topic with no home in v1.0 (rights; also on the ambiguous list).",
 "child safety": "Topic with no home in v1.0 (rights).", "profile": "Lemma of 'profiling' (rights); on the ambiguous list because of 'high-profile'.",
 "security": "Topic with no home in v1.0 (risk); on the ambiguous list ('national security').", "cybersecurity": "Topic with no home in v1.0 (risk).",
 "supply chain": "Topic with no home in v1.0 (economy).", "chip": "Topic with no home in v1.0 (economy); on the ambiguous list.",
 "semiconductor": "Topic with no home in v1.0 (economy).", "export control": "Topic with no home in v1.0 (economy).",
 "subsidy": "Topic with no home in v1.0 (economy).", "trade union": "Labour topic; plain 'union' is not used because of 'European Union'.",
 "labour union": "Labour topic.", "labor union": "Labour topic.",
 "fact check": "Fact-checking variants (misinformation).", "fact checker": "Fact-checking variants (misinformation).",
}
rows = []
for fr in FRAMES:
    for t in sorted(L0[fr] - L1[fr]): rows.append({"version": "1.0 -> 1.1", "action": "removed", "frame": fr, "term": t, "reason": REASONS.get(t)})
    for t in sorted(L1[fr] - L0[fr]): rows.append({"version": "1.0 -> 1.1", "action": "added", "frame": fr, "term": t, "reason": REASONS.get(t)})
amb0, amb1 = {lem(t) for t in v0["ambiguous_terms"]}, {lem(t) for t in v1["ambiguous_terms"]}
for t in sorted(amb1 - amb0): rows.append({"version": "1.0 -> 1.1", "action": "added to ambiguous list", "frame": "", "term": t, "reason": REASONS.get(t, "See the term's own entry.")})
for t in sorted(amb0 - amb1): rows.append({"version": "1.0 -> 1.1", "action": "removed from ambiguous list", "frame": "", "term": t, "reason": REASONS.get(t, "Term deleted from the lexicon.")})
change = pd.DataFrame(rows)
missing = change[change["reason"].isna()]
assert missing.empty, "terms changed without a reason: " + str(missing[["action", "term"]].values.tolist())
display(change.groupby(["action"]).size().to_frame("terms"))
change

,terms
action,
added,17
added to ambiguous list,4
removed,17
removed from ambiguous list,1


,version,action,frame,term,reason
0,1.0 -> 1.1,removed,innovation_opportunity,scientific progress,Sub-phrase covered by 'progress' and by longes...
1,1.0 -> 1.1,removed,innovation_opportunity,technological advancement,Sub-phrase covered by 'advancement' and by lon...
2,1.0 -> 1.1,removed,innovation_opportunity,technological progress,Sub-phrase covered by 'progress' and by longes...
3,1.0 -> 1.1,removed,risk_safety,alignment,"Often 'policy alignment', not AI alignment."
4,1.0 -> 1.1,added,risk_safety,cybersecurity,Topic with no home in v1.0 (risk).
5,1.0 -> 1.1,added,risk_safety,security,Topic with no home in v1.0 (risk); on the ambi...
6,1.0 -> 1.1,removed,regulation_governance,agency,Often 'human agency'; also appears in the Q7 p...
7,1.0 -> 1.1,removed,regulation_governance,bureaucracy,"Pejorative: measures stance, not topic (rare: ..."
8,1.0 -> 1.1,removed,regulation_governance,legal framework,Sub-phrase covered by 'framework' and by longe...
9,1.0 -> 1.1,removed,regulation_governance,licence,Second sense (data licensing).


In [3]:
# drafting clean-up before v1.0 was first shown to the team (from the project history, not recomputable from files)
drafting = ["report", "fine", "monitor", "tracking", "capital", "pioneering", "licensing", "biased", "profiling", "misleading", "civil liberties", "progress (duplicate)"]
before = {k: json.dumps(v1[k], sort_keys=True) for k in ("frames", "ambiguous_terms", "jurisdiction_terms")}

v1.setdefault("date", date.today().isoformat())                      # first tagging date is kept on reruns
v1["signed_off"] = v1.get("signed_off", False)
v1["signed_off_by"] = v1.get("signed_off_by")
v1["changelog"] = {"from": "1.0", "to": v1["version"], "entries": change.to_dict("records"),
                   "drafting_cleanup_before_v1_0": {"removed": drafting, "reason": "Too ambiguous in news text (report, fine, monitor, tracking, capital) or redundant inflected forms removed (the scorer lemmatises terms)."}}
v1["version_note"] = "v1.1 = v1.0 + the changes in this changelog. Frozen only when signed_off is true."
assert {k: json.dumps(v1[k], sort_keys=True) for k in before} == before
json.dump(v1, open(LEX_PATH, "w"), indent=2)

md = ["# Lexicon changelog", "", f"Current version: **{v1['version']}**, tagged {v1['date']}. Signed off: **{v1['signed_off']}**.", "",
      "Generated by `07_lexicon_signoff.ipynb` by comparing `reference/frame_keywords_v1.0.json` with `frame_keywords.json` on lemmas.", "",
      "| action | frame | term | reason |", "|---|---|---|---|"]
md += [f"| {r.action} | {r.frame} | {r.term} | {r.reason} |" for r in change.itertuples()]
md += ["", "Before v1.0 was first shown to the team, these terms were already dropped from the first draft: " + ", ".join(drafting[:-1]) + " (ambiguous in news text, or redundant inflected forms)."]
(FR / "CHANGELOG.md").write_text("\n".join(md) + "\n")
LEX = Lexicon(LEX_PATH)
print("lexicon tagged:", v1["version"], v1["date"], "| signed off:", v1["signed_off"], "| terms unchanged (asserted)")

lexicon tagged: 1.1 2026-10-27 | signed off: False | terms unchanged (asserted)


## 2. Do the audit tables use the post-fix numbers?
The casing fix changed news scores (2.55 → 2.77 hits per 100 words; texts under 5 hits 124 → 108). The term-count table of notebook 03 is recomputed here with the current scorer and compared with the saved file; they must match exactly.

In [4]:
ans = pd.read_json(CFG["answers_jsonl"], lines=True); ans = ans[ans["chatbot"].isin(["llama3.1:8b", "qwen3.5:9b"])].reset_index(drop=True)
news = pd.read_csv(CFG["news_corpus_csv"], keep_default_na=False)
ans["text_cut"] = ans["answer"].map(cut); news["text_cut"] = news["text"].map(cut)
H_a = find_hits(ans["text_cut"], LEX.matcher, LEX.owner); H_n = find_hits(news["text_cut"], LEX.matcher, LEX.owner)
from collections import Counter
tc_a, tc_n = Counter(l for h in H_a for _, l in h), Counter(l for h in H_n for _, l in h)

saved = pd.read_csv(OUT / "lexicon_term_counts.csv")
rec = saved[["term"]].copy(); rec["hits_chatbot"] = rec["term"].map(lambda t: tc_a.get(t, 0)); rec["hits_news"] = rec["term"].map(lambda t: tc_n.get(t, 0))
assert (rec["hits_chatbot"].to_numpy() == saved["hits_chatbot"].to_numpy()).all() and (rec["hits_news"].to_numpy() == saved["hits_news"].to_numpy()).all(), "saved term counts are stale"
sc_n = pd.read_csv(OUT / "frame_scores_news.csv")
print("term-count table matches a fresh recomputation (", len(saved), "terms, chatbot and news )")
print("news hits per 100 words:", round(sc_n["hits_per_100w"].mean(), 2), "| news texts under 5 hits:", int((sc_n["hits_total"] < 5).sum()), "| under 5 hits with entropy NaN:", int(sc_n["entropy"].isna().sum()))

term-count table matches a fresh recomputation ( 208 terms, chatbot and news )
news hits per 100 words: 2.77 | news texts under 5 hits: 108 | under 5 hits with entropy NaN: 108


## 3. Terms that never fire in the chatbot answers
Keep / drop proposals, using only evidence in the table: a term that fires in news is kept (it may matter for the news side of the comparison); a term with no hits in either corpus is a drop candidate; one or two news hits is "review". Nothing is removed from the lexicon here.

In [5]:
owner = LEX.owner; amb = LEX.ambiguous
dead = pd.DataFrame([{"frame": owner[t], "term": t, "ambiguous": t in amb, "hits_chatbot": tc_a.get(t, 0), "hits_news": tc_n.get(t, 0)} for t in owner])
dead = dead[dead["hits_chatbot"] == 0].copy()
dead["proposal"] = np.select([dead["hits_news"] >= 3, dead["hits_news"] >= 1], ["keep (fires in news)", "review (1-2 news hits)"], "drop candidate (no hits anywhere)")
dead = dead.sort_values(["proposal", "frame", "hits_news"], ascending=[True, True, False])
dead.to_csv(FR / "review_never_firing_terms.csv", index=False)
print(len(dead), "of", len(owner), "terms never fire in chatbot answers;", dead["proposal"].value_counts().to_dict())
dead

33 of 208 terms never fire in chatbot answers; {'keep (fires in news)': 17, 'drop candidate (no hits anywhere)': 9, 'review (1-2 news hits)': 7}


,frame,term,ambiguous,hits_chatbot,hits_news,proposal
174,economy_labour,labour union,False,0,0,drop candidate (no hits anywhere)
175,economy_labour,labor union,False,0,0,drop candidate (no hits anywhere)
201,misinformation_integrity,information ecosystem,False,0,0,drop candidate (no hits anywhere)
78,regulation_governance,harmonise,False,0,0,drop candidate (no hits anywhere)
116,rights_privacy,marginalise,False,0,0,drop candidate (no hits anywhere)
118,rights_privacy,vulnerable group,False,0,0,drop candidate (no hits anywhere)
120,rights_privacy,due process,False,0,0,drop candidate (no hits anywhere)
121,rights_privacy,right to explanation,False,0,0,drop candidate (no hits anywhere)
39,risk_safety,catastrophe,False,0,0,drop candidate (no hits anywhere)
168,economy_labour,supply chain,False,0,39,keep (fires in news)


## 4. Reading low-hit news chunks for missing terms
The corpus has only **24** news chunks with zero or one hit (7 with zero), so the planned sample of at least 60 zero/one-hit chunks does not exist. All 24 were read, plus **36 chunks with 2–4 hits (9 per region, seeded)** as the nearest low-density group: 60 chunks, 15 per region.

In [6]:
sc_n = pd.read_csv(OUT / "frame_scores_news.csv"); d = sc_n.merge(news[["doc_id", "text", "title"]], on="doc_id")
low = d[d["hits_total"] <= 1].assign(pool="0-1 hits (all)")
mid = d[(d["hits_total"] >= 2) & (d["hits_total"] <= 4)]
mid = pd.concat([g.sample(9, random_state=SEED) for _, g in mid.groupby("region")]).assign(pool="2-4 hits (9 per region)")
sample = pd.concat([low, mid]).sort_values(["pool", "region", "doc_id"]).reset_index(drop=True)
sample[["doc_id", "pool", "region", "outlet", "hits_total", "title", "text"]].to_csv("data/lexicon_review_lowhit_sample.csv", index=False)   # git-ignored: article text
print(len(sample), "chunks read;", sample.groupby(["pool", "region"]).size().to_dict(), "| file with the text: data/lexicon_review_lowhit_sample.csv")

60 chunks read; {('0-1 hits (all)', 'EU'): 7, ('0-1 hits (all)', 'Other'): 7, ('0-1 hits (all)', 'UK'): 4, ('0-1 hits (all)', 'US'): 6, ('2-4 hits (9 per region)', 'EU'): 9, ('2-4 hits (9 per region)', 'Other'): 9, ('2-4 hits (9 per region)', 'UK'): 9, ('2-4 hits (9 per region)', 'US'): 9} | file with the text: data/lexicon_review_lowhit_sample.csv


**Candidate terms found by reading (proposals only).** Each has the frame it would belong to, the chunk it came from, a short verbatim example (at most about 20 words), and the number of news chunks and chatbot answers that contain the term, counted by the scorer without adding the term to the lexicon. A caution column says when a term needs a decision (stance-laden, ambiguous, or corporate news rather than regulation framing).

In [7]:
CANDIDATES = [
 # term, proposed frame, doc_id, short verbatim example, caution
 ("guardrail", "risk_safety", "news_0277", "those guardrails remain woefully deficient", ""),
 ("parental control", "risk_safety", "news_0253", "add parental controls to its chatbot ChatGPT", "child-safety group; 'child' and 'child safety' are already in rights_privacy"),
 ("teenager", "risk_safety", "news_0277", "Popular chatbots missed warning signs in scenarios involving teenagers", "ambiguous topic word; consider with 'child' as one group"),
 ("suicide", "risk_safety", "news_0253", "before he died by suicide", "sensitive harm term; very rare in regulation framing"),
 ("violent", "risk_safety", "news_0277", "teenagers discussing violent acts", ""),
 ("explicit content", "risk_safety", "news_0405", "accessible to users as young as 12", "article is about sexualised chatbot content"),
 ("hate speech", "risk_safety", "news_0414", "generating inflammatory responses", "harmful-content topic"),
 ("antisemitic", "risk_safety", "news_0414", "anti-Semitic responses to questions posed by users", "topic word; spelling variants (anti-Semitic) need lemma check"),
 ("superintelligence", "risk_safety", "news_0170", "safety and alignment on Meta’s superintelligence team", ""),
 ("rogue", "risk_safety", "news_0038", "When AI Goes Rogue, Science Fiction Meets Reality", "headline word; check frequency before adding"),
 ("autonomous", "risk_safety", "news_0170", "deleted her entire inbox, ignoring instructions to pause", "also appears in product news (news_0082): ambiguous"),
 ("attacker", "risk_safety", "news_0164", "an attacker can access uploaded files, cached data", ""),
 ("lawsuit", "regulation_governance", "news_0253", "a lawsuit filed last week in a California state court", "legal enforcement; check overlap with litigation"),
 ("court", "regulation_governance", "news_0267", "A German court has ruled in favor of GEMA", "very general word"),
 ("legislative", "regulation_governance", "news_0378", "in accordance with legislative requirements", "the lexicon has legislation / legislate / legislature but not this form"),
 ("oversee", "regulation_governance", "news_0170", "a toddler that needs to be overseen", "the lexicon has oversight but not the verb"),
 ("human intervention", "regulation_governance", "news_0082", "without constant human intervention", "human-oversight idea; 'human oversight' and 'human in the loop' also candidates"),
 ("restriction", "regulation_governance", "news_0131", "a nationality-based export restriction", "ambiguous: 'ban' and 'prohibit' too general"),
 ("anonymity", "rights_privacy", "news_0300", "anonymity is the only protection for the outcast", ""),
 ("censorship", "rights_privacy", "news_0304", "The Hidden Hand Behind AI Content Filters", "free-expression topic; 'content filter' is a candidate too"),
 ("private data", "rights_privacy", "news_0164", "access your private data", "phrase; 'personal data' is already in the lexicon"),
 ("patent", "rights_privacy", "news_0250", "AI use in the patent profession has become a rushing torrent", "intellectual property; or innovation? needs a decision"),
 ("acquisition", "economy_labour", "news_0213", "SpaceX has acquired his artificial-intelligence startup xAI", "corporate news, not regulation framing"),
 ("valuation", "economy_labour", "news_0105", "Harvey, a leading legal AI company, is now valued at $11 billion", "corporate news, not regulation framing"),
 ("plagiarism", "misinformation_integrity", "news_0040", "pass it off as your own", "academic integrity; 'cheating' and 'academic dishonesty' also candidates"),
 ("academic dishonesty", "misinformation_integrity", "news_0040", "what constitutes academic dishonesty", ""),
 ("inaccurate", "misinformation_integrity", "news_0261", "incorrect evidence garnered from Microsoft Copilot", "also 'incorrect', 'accuracy'; overlaps hallucination"),
 ("problematic", "misinformation_integrity", "news_0279", "five popular chatbots give “problematic” answers to health-related queries", "very general word"),
 ("echo chamber", "misinformation_integrity", "news_0086", "Escape your echo chamber", "this example is newsletter boilerplate, not article text: low value"),
 ("news ecosystem", "misinformation_integrity", "news_0134", "an important part of the Australian news ecosystem", "the lexicon has 'information ecosystem'"),
 ("sycophancy", "risk_safety", "news_0324", "ChatGPT always says: ‘Oh, your questions are clever.", "or misinformation_integrity; needs a decision"),
 ("political bias", "misinformation_integrity", "news_0265", "treat opposing political viewpoints with equal depth", "'bias' alone was removed in v1.1 (second meaning); phrases 'political bias', 'algorithmic bias', 'implicit bias' could be tested. Do NOT add 'woke' (stance-laden)"),
]
cand = pd.DataFrame(CANDIDATES, columns=["term", "proposed_frame", "doc_id", "example", "caution"])
by_frame = {}
for t, f in zip(cand["term"], cand["proposed_frame"]): by_frame.setdefault(f, []).append(t)
cm, co = build_matcher(by_frame)
Hc_n = find_hits(news["text_cut"], cm, co); Hc_a = find_hits(ans["text_cut"], cm, co)
cn, ca = Counter(l for h in Hc_n for _, l in set(h)), Counter(l for h in Hc_a for _, l in set(h))
cand["news_chunks_with_term"] = cand["term"].map(lambda t: cn.get(lem(t), 0)); cand["answers_with_term"] = cand["term"].map(lambda t: ca.get(lem(t), 0))
# every example sentence must really occur in the cited chunk
txt = dict(zip(sample["doc_id"], sample["text"] + " " + sample["title"]))
norm_ws = lambda s: re.sub(r"\s+", " ", s)
bad = [(r.term, r.doc_id) for r in cand.itertuples() if norm_ws(r.example).lower() not in norm_ws(txt[r.doc_id]).lower()]
print("examples not found verbatim in the cited chunk:", bad)
cand.to_csv("data/lexicon_review_candidates_with_examples.csv", index=False)          # local only: holds short quotes from articles
cand.drop(columns=["example"]).to_csv(FR / "review_candidate_terms.csv", index=False)  # committed: terms, frames, counts and doc ids, no quotes
cand.drop(columns=["example"])

examples not found verbatim in the cited chunk: []


,term,proposed_frame,doc_id,caution,news_chunks_with_term,answers_with_term
0,guardrail,risk_safety,news_0277,,18,8
1,parental control,risk_safety,news_0253,child-safety group; 'child' and 'child safety'...,2,0
2,teenager,risk_safety,news_0277,ambiguous topic word; consider with 'child' as...,3,0
3,suicide,risk_safety,news_0253,sensitive harm term; very rare in regulation f...,16,0
4,violent,risk_safety,news_0277,,4,1
5,explicit content,risk_safety,news_0405,article is about sexualised chatbot content,2,0
6,hate speech,risk_safety,news_0414,harmful-content topic,3,0
7,antisemitic,risk_safety,news_0414,topic word; spelling variants (anti-Semitic) n...,3,0
8,superintelligence,risk_safety,news_0170,,2,1
9,rogue,risk_safety,news_0038,headline word; check frequency before adding,1,0


**Other findings from the reading (corpus quality, not lexicon).** Boilerplate (subscription banners, newsletter sign-ups, navigation text), leftover HTML and garbled characters appear inside article text. They dilute frame density in the 300-word head. Measured on the whole corpus below.

In [8]:
heads = news["text_cut"]
checks = {"subscription / sign-up / membership boilerplate": r"(?i)subscribe now|sign up for|newsletter|become an independent member|already a member|log in|enjoy our content|signup to continue",
          "leftover HTML tags": r"<\s*/?\s*[a-z][a-z0-9]*[^>]*>", "garbled characters (mojibake)": r"â€|Ã.|â\s",
          "dateline / share-widget text": r"(?i)sharelink copied|share this article|published by [a-z.]+ \d|^\s*by [A-Z][a-z]+ [A-Z][a-z]+"}
qual = pd.DataFrame([{"issue": k, "chunks_affected": int(heads.str.contains(p, regex=True).sum()), "share": round(heads.str.contains(p, regex=True).mean(), 3)} for k, p in checks.items()])

# head/body mismatch: the on-topic rule was applied to the full text, the scorer sees only the first 300 words
AI_WORDS  = re.compile(r"\b(AI|artificial intelligence|chatbots?|ChatGPT|large language models?|LLMs?|OpenAI|Gemini)\b")
REG_WORDS = re.compile(r"\b(regulat\w*|legislat\w*|laws?|AI Act|bills?|rules?|oversight|ban|governance|policy|lawmakers?)\b", re.I)
on_head = heads.map(lambda t: len(AI_WORDS.findall(t)) >= 2 and len(REG_WORDS.findall(t)) >= 2)
AMBIG_REG = re.compile(r"\b(laws?|bills?|ban|rules?|policy)\b", re.I)
strict_reg = news["text"].map(lambda t: len([m for m in REG_WORDS.findall(t) if not AMBIG_REG.fullmatch(m)]))
qual.loc[len(qual)] = ["corpus articles that still satisfy the on-topic rule within their first 300 words", int(on_head.sum()), round(on_head.mean(), 3)]
qual.loc[len(qual)] = ["articles passing the regulation test only thanks to the ambiguous words (law, bill, ban, rule, policy) in the FULL text", int((strict_reg < 2).sum()), round((strict_reg < 2).mean(), 3)]
qual.to_csv(FR / "review_corpus_quality.csv", index=False)
qual

,issue,chunks_affected,share
0,subscription / sign-up / membership boilerplate,15,0.036
1,leftover HTML tags,1,0.002
2,garbled characters (mojibake),3,0.007
3,dateline / share-widget text,11,0.026
4,corpus articles that still satisfy the on-topi...,157,0.374
5,articles passing the regulation test only than...,161,0.383


## 5. Weak terms, prompt-echo terms and the on-topic rule

In [9]:
# prompt-echo: for each term, the share of its chatbot hits that sit in answers whose own prompt contains the term
prompts = pd.read_csv(CFG["prompts_csv"]).set_index("prompt_id")["prompt"]
pt = {pid: {l for _, l in h} for pid, h in zip(prompts.index, find_hits(prompts.tolist(), LEX.matcher, LEX.owner))}
tot, echo = Counter(), Counter()
for h, pid in zip(H_a, ans["prompt_id"]):
    for _, l in h:
        tot[l] += 1; echo[l] += l in pt[pid]
echo_tab = pd.DataFrame([{"frame": owner[t], "term": t, "hits_chatbot": n, "hits_in_answers_whose_prompt_has_the_term": echo[t], "echo_share": round(echo[t] / n, 2)} for t, n in tot.items()])
echo_tab = echo_tab[(echo_tab["hits_chatbot"] >= 15) & (echo_tab["echo_share"] >= 0.4)].sort_values("echo_share", ascending=False)
echo_tab.to_csv(FR / "review_prompt_echo_terms.csv", index=False)
print("prompt-echo terms (at least 15 chatbot hits and at least 40% of them in answers whose prompt contains the term):")
echo_tab

prompt-echo terms (at least 15 chatbot hits and at least 40% of them in answers whose prompt contains the term):


,frame,term,hits_chatbot,hits_in_answers_whose_prompt_has_the_term,echo_share
59,innovation_opportunity,start up,42,42,1.00
107,rights_privacy,human review,38,38,1.00
97,rights_privacy,personal datum,34,33,0.97
53,economy_labour,worker,36,34,0.94
22,rights_privacy,facial recognition,50,45,0.90
38,regulation_governance,approval,54,45,0.83
77,misinformation_integrity,false,57,37,0.65
73,risk_safety,high stake,28,14,0.50
58,regulation_governance,regulator,70,32,0.46
39,economy_labour,market,87,36,0.41


In [10]:
weak = ["transform", "creative", "experiment", "unlock", "dominate", "dominance", "progress", "potential", "benefit"]
weak_tab = pd.DataFrame([{"term": t, "frame": owner[lem(t)], "on_ambiguous_list": lem(t) in amb, "hits_chatbot": tc_a.get(lem(t), 0), "hits_news": tc_n.get(lem(t), 0),
                          "chatbot_hits_in_prompt_echo": echo[lem(t)]} for t in weak])
weak_tab

,term,frame,on_ambiguous_list,hits_chatbot,hits_news,chatbot_hits_in_prompt_echo
0,transform,innovation_opportunity,False,2,39,0
1,creative,innovation_opportunity,False,9,19,0
2,experiment,innovation_opportunity,False,6,23,0
3,unlock,innovation_opportunity,False,0,20,0
4,dominate,economy_labour,False,14,6,12
5,dominance,economy_labour,False,13,6,0
6,progress,innovation_opportunity,True,79,13,0
7,potential,innovation_opportunity,True,161,72,0
8,benefit,innovation_opportunity,True,109,31,0


## 6. The AI-but-not-regulation articles
The on-topic rule needs at least 2 AI words and at least 2 regulation words. Articles with AI words but fewer than 2 regulation words were rejected. They are collected in one local file (it holds article text, so it is git-ignored) so the team can read them in one sitting and answer the question below. The corpus is frozen at 420: the answer affects the limitations section and any future corpus version, not the current data.

In [11]:
scraped = pd.read_csv(CFG["scraped_csv"], keep_default_na=False); urls = pd.read_csv(CFG["urls_full_csv"])
pool = scraped.merge(urls[["url", "outlet", "region"]], on="url")
pool = pool[(pool["status"] == "ok") & (pool["n_words"].astype(float) <= 3000)].copy()
pool["ai_hits"] = pool["text"].map(lambda t: len(AI_WORDS.findall(t))); pool["reg_hits"] = pool["text"].map(lambda t: len(REG_WORDS.findall(t)))
rejected = pool[~((pool["ai_hits"] >= 2) & (pool["reg_hits"] >= 2))]
ai_only = rejected[(rejected["ai_hits"] >= 2) & (rejected["reg_hits"] < 2)]
print(len(ai_only), "articles have 2+ AI words but fewer than 2 regulation words; by region:", ai_only["region"].value_counts().to_dict())
pick = ai_only.sample(min(30, len(ai_only)), random_state=SEED).sort_values(["region", "outlet"])
lines = ["# AI-but-not-regulation articles (30, seeded, read in one sitting)", "",
 f"Pool: {len(ai_only)} usable articles with at least 2 AI words and fewer than 2 regulation words (the rule keeps articles with at least 2 of each). Local file: contains article text, git-ignored.", "",
 "## The question",
 "For each article below: does it carry regulation, governance, rights, safety or similar framing that the regulation word list (regulat*, legislat*, law(s), AI Act, bill(s), rule(s), oversight, ban, governance, policy, lawmaker(s)) missed? Mark Y / N / unsure.",
 "- If most are **N** (consumer tech, business news, education): the rule is doing its job; keep it and describe it in the limitations.",
 "- If many are **Y** and the missing words are things like *guardrails, safeguards, restrictions, compliance, court, lawsuit, liability*: the regulation list is too narrow; a future corpus version should widen it (the current corpus stays frozen).", ""]
for r in pick.itertuples():
    lines += [f"## {r.region} | {r.outlet} | AI words {r.ai_hits}, regulation words {r.reg_hits}", f"{' '.join(r.text.split()[:130])} ...", f"<{r.url}>", ""]
Path("data/review_ai_not_regulation.md").write_text("\n".join(lines))
print("written: data/review_ai_not_regulation.md (", len(pick), "articles )")

988 articles have 2+ AI words but fewer than 2 regulation words; by region: {'Other': 298, 'UK': 264, 'EU': 214, 'US': 212}
written: data/review_ai_not_regulation.md ( 30 articles )


## 7. The review file (nothing here is merged into the lexicon)
`dataset/frames/lexicon_review.md` is committed and contains no article text. `data/lexicon_review_with_examples.md` is the same file plus the short example sentences (git-ignored).

In [12]:
def table(df, cols):
    return ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)] + ["| " + " | ".join(str(r[c]) for c in cols) + " |" for _, r in df.iterrows()]

def review(with_examples):
    L = ["# Lexicon v1.1 – proposed changes for review", "",
         f"Lexicon tag: v{v1['version']}, {v1['date']}, signed off: **{v1['signed_off']}**. **Nothing in this file has been applied to `frame_keywords.json`.** No H1–H3 test has been run.", "",
         "How to use: tick or strike each line, then tell the assistant to apply the approved ones as v1.2 (new changelog entries) or sign v1.1 off as it is.", "",
         "## A. Terms that never fire in the chatbot answers (" + str(len(dead)) + " of " + str(len(owner)) + ")", "",
         "Proposal rule: keep if it fires in news (3 or more hits); review if 1–2 news hits; drop candidate if no hits anywhere.", ""]
    L += table(dead, ["frame", "term", "hits_news", "proposal"])
    L += ["", "## B. Candidate missing terms found by reading 60 low-hit news chunks", "",
          "Read by the AI assistant (all 24 chunks with 0–1 hits plus 36 with 2–4 hits, 15 per region). Counts are chunks/answers containing the term. Not exhaustive; to be confirmed by a human reader.", ""]
    cols = ["term", "proposed_frame", "news_chunks_with_term", "answers_with_term", "caution"] + (["doc_id", "example"] if with_examples else [])
    L += table(cand, cols)
    L += ["", "Observations: most 0–1 hit chunks are consumer-tech pieces, essays or school stories, not regulation coverage; the missing-term list is short and most candidates fire in only a few chunks. Corpus-quality counts below.", "",
          "## C. Corpus quality seen while reading (affects frame density, not the lexicon)", ""] + table(qual, ["issue", "chunks_affected", "share"])
    L += ["", "## D. Existing terms to reconsider", "", "Weak or prompt-linked terms (counts from chatbot answers and news):", ""] + table(weak_tab, ["term", "frame", "on_ambiguous_list", "hits_chatbot", "hits_news", "chatbot_hits_in_prompt_echo"])
    L += ["", "Prompt-echo terms (at least 15 chatbot hits, at least 40% of them in answers whose prompt contains the term):", ""] + table(echo_tab, ["frame", "term", "hits_chatbot", "echo_share"])
    L += ["", "## E. On-topic rule (corpus is frozen; this concerns limitations and any future version)", "",
          "- The rule was applied to the full text but frames are scored on the first 300 words: see the share of articles that still satisfy the rule within the head (section C).",
          f"- {len(ai_only)} usable articles were rejected for having AI words but fewer than 2 regulation words. 30 are in `data/review_ai_not_regulation.md` with the question to answer (is the regulation word list too narrow?).",
          "- Some regulation words (law, bill, ban, rule, policy) are everyday words; section C counts articles that pass only through them.", ""]
    return "\n".join(L)

(FR / "lexicon_review.md").write_text(review(False)); Path("data/lexicon_review_with_examples.md").write_text(review(True))
print("written: dataset/frames/lexicon_review.md and data/lexicon_review_with_examples.md")

written: dataset/frames/lexicon_review.md and data/lexicon_review_with_examples.md


## 8. Confirm that no hypothesis test has been run

In [13]:
import glob
forbidden = glob.glob(str(OUT / "h1_*.csv")) + glob.glob(str(OUT / "style_matched_entropy_comparison.csv"))
assert not forbidden, "an H1 result file exists: " + str(forbidden)
for f in ["05_rarefied_entropy.ipynb", "06_style_vs_framing.ipynb"]:
    src = "".join("".join(c["source"]) for c in json.load(open(f))["cells"] if c["cell_type"] == "code")
    assert "H_TESTS_ENABLED = False" in src, f"the gate in {f} is open"
print("no H1 result files; the gates in notebooks 05 and 06 are closed; lexicon signed off:", json.load(open(LEX_PATH))["signed_off"])

no H1 result files; the gates in notebooks 05 and 06 are closed; lexicon signed off: False
